## NER & SA

In [1]:
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

In [2]:
content = "Trinamool Congress leader Mahua Moitra has moved the Supreme Court against her expulsion from the Lok Sabha over the cash-for-query allegations against her. Moitra was ousted from the Parliament last week after the Ethics Committee of the Lok Sabha found her guilty of jeopardising national security by sharing her parliamentary portal's login credentials with businessman Darshan Hiranandani."

In [4]:
doc = nlp(content)

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Trinamool Congress 0 18 ORG
Mahua Moitra 26 38 PERSON
the Supreme Court 49 66 ORG
Moitra 157 163 NORP
Parliament 184 194 ORG
last week 195 204 DATE
the Ethics Committee 211 231 ORG
Darshan Hiranandani 373 392 PERSON


In [5]:
from spacy import displacy

displacy.render(doc, style='ent')

In [7]:
entities = [(ent.text, ent.label_, ent.lemma_) for ent in doc.ents]
df = pd.DataFrame(entities, columns=['text', 'type', 'lemma'])
print(df)

                   text    type                 lemma
0    Trinamool Congress     ORG    Trinamool Congress
1          Mahua Moitra  PERSON          Mahua Moitra
2     the Supreme Court     ORG     the Supreme Court
3                Moitra    NORP                Moitra
4            Parliament     ORG            Parliament
5             last week    DATE             last week
6  the Ethics Committee     ORG  the Ethics Committee
7   Darshan Hiranandani  PERSON   Darshan Hiranandani


In [19]:
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [9]:
text = "TextBlob is very amazing and simple to use. What a great tool!"

In [11]:
blob = TextBlob(text)

In [18]:
print(blob.words)
print(blob.sentences)
print(blob.tags)
print(blob.noun_phrases)

['TextBlob', 'is', 'very', 'amazing', 'and', 'simple', 'to', 'use', 'What', 'a', 'great', 'tool']
[Sentence("TextBlob is very amazing and simple to use."), Sentence("What a great tool!")]
[('TextBlob', 'NNP'), ('is', 'VBZ'), ('very', 'RB'), ('amazing', 'JJ'), ('and', 'CC'), ('simple', 'JJ'), ('to', 'TO'), ('use', 'VB'), ('What', 'WP'), ('a', 'DT'), ('great', 'JJ'), ('tool', 'NN')]
['textblob', 'great tool']


In [16]:
print(blob.sentiment)

Sentiment(polarity=0.5933333333333334, subjectivity=0.7023809523809524)


Polarity - Range: -1 to +1

Subjectivity - Fact/opinionated - Range: 0 to 1

In [22]:
sid_obj = SentimentIntensityAnalyzer()
sentiment_dict = sid_obj.polarity_scores(text)
sentiment_dict

{'neg': 0.149, 'neu': 0.851, 'pos': 0.0, 'compound': -0.4215}

In [35]:

text = "TextBlob is very amazing and simple to use. What a great tool!"
text = "It happened yesterday night when we were going back to home after office, I guess they were around 5 people. Suddenly they came and started attacking us. I really got frightened.It was such a bad experience for me"

sentiment_dict = sid_obj.polarity_scores(text)
blob = TextBlob(text)

blob.sentiment
sentiment_dict

{'neg': 0.153, 'neu': 0.847, 'pos': 0.0, 'compound': -0.7579}

compound - normalized overall sentiment score (-1 to +1)

## BERT & GPT

In [37]:
df = pd.read_csv("../../data/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [39]:
df['review_cleaned'] = df['review'].apply(lambda x: x.replace('<br />', ''))

In [42]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification


In [43]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# take one review
review = df['review'].iloc[0]

# tokenize
inputs = tokenizer(review, return_tensors="pt", truncation=True, padding=True)

# model prediction
outputs = model(**inputs)
logits = outputs.logits
pred = torch.argmax(logits, dim=1)

print("Review:", review[:120])
print("Predicted label:", pred.item())

In [44]:
#fine tune

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# load dataset
df = pd.read_csv("../../data/IMDB Dataset.csv")

# take small subset for demo
df = df.sample(500)

# convert labels
df["label"] = df["sentiment"].map({"positive":1, "negative":0})

dataset = Dataset.from_pandas(df[["review","label"]])

# tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["review"], truncation=True, padding="max_length", max_length=256)

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])

# model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# training setup
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/home/asish-jose/AI-ML/venv/lib/pytho

Step,Training Loss


KeyboardInterrupt: 

### GPT

In [45]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [48]:
print(torch.cuda.is_available())

device = torch.device("cpu")
model.to(device)

False


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
prompt = "Artitificial Intelligence is"

inputs = tokenizer(prompt, return_tensors='pt').to(device)

output = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,  #greedy sampling
    temperature=1.2, #controls randomness by scaling logits before softmax
    top_k=50,        #keeps only top k most probable tokens
    top_p=0.9,        #nucleus sampling
    num_return_sequences=3, #generates multiple outputs
    #num_beams=5,        #beam search (often used for translation/summarization)
)

out = tokenizer.decode(output2[0], skip_special_tokens=True)
print(out)

## lstm

In [51]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


with open("../../data/quotes/train.txt") as f:
    text = f.read().lower()[:20000]   # take only small portion
text

'i\'m selfish, impatient and a little insecure. i make mistakes, i am out of control and at times hard to handle. but if you can\'t handle me at my worst, then you sure as hell don\'t deserve me at my best.\nyou\'ve gotta dance like there\'s nobody watching,love like you\'ll never be hurt,sing like there\'s nobody listening,and live like it\'s heaven on earth.\nyou know you\'re in love when you can\'t fall asleep because reality is finally better than your dreams.\na friend is someone who knows all about you and still loves you.\ndarkness cannot drive out darkness: only light can do that. hate cannot drive out hate: only love can do that.\nwe accept the love we think we deserve.\nonly once in your life, i truly believe, you find someone who can completely turn your world around. you tell them things that you’ve never shared with another soul and they absorb everything you say and actually want to hear more. you share hopes for the future, dreams that will never come true, goals that we

In [ ]:
words = text.lower().split()

# build vocab
vocab = sorted(set(words))
stoi = {w:i for i,w in enumerate(vocab)}
vocab_size = len(vocab)

# encode words
data = torch.tensor([stoi[w] for w in words])

# create input-target sequences
seq_len = 10
X, Y = [], []

for i in range(len(data)-seq_len):
    X.append(data[i:i+seq_len])
    Y.append(data[i+1:i+seq_len+1])

X = torch.stack(X)
Y = torch.stack(Y)

loader = DataLoader(TensorDataset(X,Y), batch_size=32)

In [56]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 128, batch_first=True)
        self.fc = nn.Linear(128, vocab_size)

    def forward(self,x):
        x = self.embed(x)
        out,_ = self.lstm(x)
        return self.fc(out)

model = LSTMModel()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)


for Xb,Yb in loader:
    opt.zero_grad()
    out = model(Xb)
    loss = loss_fn(out.view(-1,vocab_size), Yb.view(-1))
    loss.backward()
    opt.step()
    break   

## Translation model

In [ ]:
class TranslationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.src_emb = nn.Embedding(len(src_vocab),32)
        self.tgt_emb = nn.Embedding(len(tgt_vocab),32)

        self.transformer = nn.Transformer(
            d_model=32,
            nhead=2,
            num_encoder_layers=1,
            num_decoder_layers=1
        )

        self.fc = nn.Linear(32,len(tgt_vocab))

    def forward(self,src,tgt):
        src = self.src_emb(src).permute(1,0,2)
        tgt = self.tgt_emb(tgt).permute(1,0,2)

        out = self.transformer(src,tgt)
        return self.fc(out)
    

model = TranslationModel()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters())